In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df=pd.read_csv('NER_raw.csv')

**Checking the Structure of the Data**

In [3]:
print(df.head())
print(df.isna().sum())
print(df['Tag'].value_counts())

    Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1          NaN             of   IN   O
2          NaN  demonstrators  NNS   O
3          NaN           have  VBP   O
4          NaN        marched  VBN   O
Sentence #    1000616
Word               10
POS                 0
Tag                 0
dtype: int64
Tag
O        887908
B-geo     37644
B-tim     20333
B-org     20143
I-per     17251
B-per     16990
I-org     16784
B-gpe     15870
I-geo      7414
I-tim      6528
B-art       402
B-eve       308
I-art       297
I-eve       253
B-nat       201
I-gpe       198
I-nat        51
Name: count, dtype: int64


**Filling the NAN values**

In [4]:
df = df.ffill()


**Grouping words and tags by sentence**

In [5]:
from collections import defaultdict

sentences=[]
labels=[]

grouped=df.groupby('Sentence #')

for _,group in grouped:
    words=list(group['Word'])
    tags=list(group['Tag'])
    sentences.append(words)
    labels.append(tags)

**Encode Labels (NER Tags)**

In [6]:
from sklearn.preprocessing import LabelEncoder


le=LabelEncoder()

# Flatten all the tags into one big list
all_tags=list(set(tag for seq in labels for tag in seq))
le.fit(all_tags)

# Build two dictionaries to map back and forth
label2id = {tag: idx for idx, tag in enumerate(le.classes_)}
id2label = {idx: tag for tag, idx in label2id.items()}

**Tokenization & Label Alignment**

In [7]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained('bert-base-cased')

def tokenize_and_align_labels(sentences, labels):
    tokenized_inputs = tokenizer(
        sentences,
        is_split_into_words=True,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    aligned_labels = []
    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                # You could use the same label or 'I-' version
                label_ids.append(label2id[label[word_idx]])
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

**Createing a Dataset Class**

In [8]:
from torch.utils.data import Dataset

class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])


**Train_test_split**

In [9]:
from sklearn.model_selection import train_test_split

train_sentences, val_sentences, train_labels, val_labels = train_test_split(
    sentences, labels, test_size=0.1, random_state=42
)

train_encodings = tokenize_and_align_labels(train_sentences, train_labels)
val_encodings = tokenize_and_align_labels(val_sentences, val_labels)

train_dataset = NERDataset(train_encodings)
val_dataset = NERDataset(val_encodings)


**Loading the Model to Start training**

In [11]:
from transformers import BertForTokenClassification , Trainer , TrainingArguments ,DataCollatorForTokenClassification
import os

os.environ["WANDB_DISABLED"] = "true"

data_collator = DataCollatorForTokenClassification(tokenizer)

model=BertForTokenClassification.from_pretrained(
    'bert-base-cased',
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=500,     # log every 500 steps
    save_steps=1000,       # save every 1000 steps
    save_total_limit=1     # keep only 1 checkpoint
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=None,
    data_collator=data_collator
)

trainer.train()


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipykernel_35/1325049231.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.285800
1000,0.161400
1500,0.144900
2000,0.139900
2500,0.132900
3000,0.115200
3500,0.105600
4000,0.107000
4500,0.104500
5000,0.103500


TrainOutput(global_step=8094, training_loss=0.11887070877569117, metrics={'train_runtime': 2210.6117, 'train_samples_per_second': 58.576, 'train_steps_per_second': 3.661, 'total_flos': 1.0310516605300392e+16, 'train_loss': 0.11887070877569117, 'epoch': 3.0})

**Evaluation**

In [13]:
!pip install seqeval


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=4a6fc6f3d402ea2fe012ced41b0e5b60e9bdbc03b079de8f7507818f297015cb
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [15]:
from seqeval.metrics import classification_report

predictions = preds.predictions.argmax(-1)
label_ids = preds.label_ids

true_labels = []
pred_labels = []

for pred, label in zip(predictions, label_ids):
    true_seq = []
    pred_seq = []
    for p, l in zip(pred, label):
        if l != -100:
            true_seq.append(id2label[l])
            pred_seq.append(id2label[p])
    true_labels.append(true_seq)
    pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))


              precision    recall  f1-score   support

         art       0.17      0.09      0.12        92
         eve       0.26      0.21      0.24        42
         geo       0.86      0.91      0.88      6167
         gpe       0.95      0.93      0.94      1790
         nat       0.57      0.17      0.26        24
         org       0.76      0.72      0.74      3656
         per       0.80      0.83      0.81      2782
         tim       0.87      0.87      0.87      2229

   micro avg       0.84      0.84      0.84     16782
   macro avg       0.65      0.59      0.61     16782
weighted avg       0.83      0.84      0.84     16782



**Inference on New Text**

In [21]:
import torch
model.to("cuda" if torch.cuda.is_available() else "cpu")


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [22]:
def predict(text):
    tokens = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt")
    tokens = {k: v.to(model.device) for k, v in tokens.items()}  # move to same device as model

    outputs = model(**tokens)
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    word_ids = tokens['input_ids'].cpu().squeeze().tolist()  # word_ids must be computed on CPU tokenizer

    word_map = tokenizer(text.split(), is_split_into_words=True).word_ids()
    final_predictions = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_map):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            label = id2label[predictions[idx]]
            final_predictions.append((text.split()[word_idx], label))
        previous_word_idx = word_idx

    return final_predictions


In [23]:
# Example 1
print(predict("Barack Obama visited Egypt in 2010."))

# Example 2
print(predict("Apple Inc. is based in California."))


[('Barack', 'B-per'), ('Obama', 'I-per'), ('visited', 'O'), ('Egypt', 'B-geo'), ('in', 'O'), ('2010.', 'B-tim')]
[('Apple', 'B-org'), ('Inc.', 'I-org'), ('is', 'O'), ('based', 'O'), ('in', 'O'), ('California.', 'B-geo')]
